<a href="https://colab.research.google.com/github/japhia16/AI-Assisted-Underwriting-Triage-Agent/blob/main/student1_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

pip install xgboost shap statsmodels

In [ ]:
!pip install pandas numpy scikit-learn openpyxl

In [ ]:
import pandas as pd

df = pd.read_csv('/content/synthetic_commercial_submissions_india.csv')
display(df.head())

,Submission_ID,District,Industry_Type,Occupancy,Construction_Type,Building_Age_Years,Sum_Insured_INR,Number_of_Employees,Years_in_Business,Prior_Claims_Count,Sprinkler_System,Fire_Hydrant_Onsite,Deductible_INR,Requested_Coverage,Risk_Score_Raw,Risk_Label
0,SUB-2024000001,Kamrup Metropolitan,Textiles & Garments,Warehouse,Brick & Mortar,9,36702000,75,44.0,0,Yes,Yes,50000,Standard Fire & Special Perils,0.6575,Medium
1,SUB-2024000002,Ranga Reddy,Plastics & Packaging,Hotel / Resort,Brick & Mortar,14,7219000,769,39.0,0,Yes,Yes,500000,All Risk (Comprehensive),0.5630,Low
2,SUB-2024000003,Debagarh,Paper & Printing,Hotel / Resort,RCC (Reinforced Cement Concrete),35,4017000,220,17.0,0,No,No,100000,Standard Fire & Special Perils,0.6214,Medium
3,SUB-2024000004,Gwalior,Educational Institution,Hospital / Clinic,RCC (Reinforced Cement Concrete),4,39304000,111,16.0,0,No,No,500000,Standard Fire & Special Perils,0.4760,Low
4,SUB-2024000005,Lakshadweep,Warehousing & Logistics,Hotel / Resort,RCC (Reinforced Cement Concrete),4,33075000,931,44.0,1,No,Yes,50000,Fire + Burglary,0.5337,Low


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Submission_ID        5000 non-null   object 
 1   District             5000 non-null   object 
 2   Industry_Type        5000 non-null   object 
 3   Occupancy            4800 non-null   object 
 4   Construction_Type    4850 non-null   object 
 5   Building_Age_Years   5000 non-null   int64  
 6   Sum_Insured_INR      5000 non-null   int64  
 7   Number_of_Employees  5000 non-null   int64  
 8   Years_in_Business    4900 non-null   float64
 9   Prior_Claims_Count   5000 non-null   int64  
 10  Sprinkler_System     4750 non-null   object 
 11  Fire_Hydrant_Onsite  5000 non-null   object 
 12  Deductible_INR       5000 non-null   int64  
 13  Requested_Coverage   5000 non-null   object 
 14  Risk_Score_Raw       5000 non-null   float64
 15  Risk_Label           5000 non-null   o

In [ ]:
hazard = pd.read_csv("india_catastrophic_risk_with_severity_scores.csv")

# Rename columns for clean merging
hazard = hazard.rename(columns={
    "Flood Risk Score":      "Flood_Risk_Score",
    "Cyclone Risk Score":    "Cyclone_Risk_Score",
    "Rainfall Risk Score":   "Rainfall_Risk_Score",
    "Earthquake Risk Score": "Earthquake_Risk_Score"
})

# Keep only what's needed
hazard = hazard[["District", "Flood_Risk_Score", "Cyclone_Risk_Score",
                 "Rainfall_Risk_Score", "Earthquake_Risk_Score"]]

# Merge onto submissions by district
df = df.merge(hazard, on="District", how="left")

# Check how many matched
print("Rows after merge:", len(df))
print("Unmatched districts:", df["Flood_Risk_Score"].isnull().sum())

# Fill any unmatched with median
for col in ["Flood_Risk_Score", "Cyclone_Risk_Score",
            "Rainfall_Risk_Score", "Earthquake_Risk_Score"]:
    df[col] = df[col].fillna(df[col].median())

print()
print("Hazard columns added successfully.")
df[["District", "Flood_Risk_Score", "Cyclone_Risk_Score",
    "Rainfall_Risk_Score", "Earthquake_Risk_Score"]].head(5)

Rows after merge: 5025
Unmatched districts: 988

Hazard columns added successfully.


,District,Flood_Risk_Score,Cyclone_Risk_Score,Rainfall_Risk_Score,Earthquake_Risk_Score
0,Kamrup Metropolitan,25.0,5.0,20.0,20.0
1,Ranga Reddy,16.0,4.0,16.0,9.0
2,Debagarh,16.0,4.0,16.0,9.0
3,Gwalior,6.0,2.0,4.0,6.0
4,Lakshadweep,20.0,20.0,12.0,8.0


In [ ]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("Total missing:", df.isnull().sum().sum())

Missing values per column:
Occupancy            202
Construction_Type    152
Years_in_Business    100
Sprinkler_System     250
dtype: int64

Total missing: 704


In [ ]:
# Categorical columns -- fill with "Unknown"
cat_cols = ["Occupancy", "Construction_Type", "Industry_Type",
            "Sprinkler_System", "Fire_Hydrant_Onsite", "Requested_Coverage"]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# Numeric columns -- fill with median
num_cols = ["Building_Age_Years", "Sum_Insured_INR", "Number_of_Employees",
            "Years_in_Business", "Prior_Claims_Count", "Deductible_INR"]

for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Confirm no more missing values in these columns
print("Missing values after filling:")
print(df[cat_cols + num_cols].isnull().sum())
print()
print("All zeros means no missing values remaining.")

Missing values after filling:
Occupancy              0
Construction_Type      0
Industry_Type          0
Sprinkler_System       0
Fire_Hydrant_Onsite    0
Requested_Coverage     0
Building_Age_Years     0
Sum_Insured_INR        0
Number_of_Employees    0
Years_in_Business      0
Prior_Claims_Count     0
Deductible_INR         0
dtype: int64

All zeros means no missing values remaining.


In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd
import numpy as np
import pickle

cat_cols = ["Occupancy", "Construction_Type", "Industry_Type",
            "Sprinkler_System", "Fire_Hydrant_Onsite", "Requested_Coverage"]

num_cols = ["Building_Age_Years", "Sum_Insured_INR", "Number_of_Employees",
            "Years_in_Business", "Prior_Claims_Count", "Deductible_INR"]

# Only keep columns that actually exist
cat_cols = [c for c in cat_cols if c in df.columns]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(df[cat_cols])

encoded_array = encoder.transform(df[cat_cols])
encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoder.get_feature_names_out(cat_cols),
    index=df.index
)

print("Encoding done.")
print("Encoded columns:", encoded_df.shape[1])

Encoding done.
Encoded columns: 45


In [ ]:
# Drop original text columns
df = df.drop(columns=cat_cols)

# Add the new encoded columns
df = pd.concat([df, encoded_df], axis=1)

print("Shape after encoding:", df.shape)
print("Text columns removed, numeric encoded columns added.")

Shape after encoding: (5025, 59)
Text columns removed, numeric encoded columns added.


In [ ]:
num_cols = [c for c in num_cols if c in df.columns]

scaler = StandardScaler()
scaler.fit(df[num_cols])
df[num_cols] = scaler.transform(df[num_cols])

print("Numeric columns scaled.")
print()
print("After scaling, stats should be close to mean=0, std=1:")
print(df[num_cols].describe().round(2))

Numeric columns scaled.

After scaling, stats should be close to mean=0, std=1:
       Building_Age_Years  Sum_Insured_INR  Number_of_Employees  \
count             5025.00          5025.00              5025.00   
mean                -0.00            -0.00                 0.00   
std                  1.00             1.00                 1.00   
min                 -1.69            -0.86                -1.72   
25%                 -0.85            -0.60                -0.85   
50%                 -0.01            -0.31                -0.00   
75%                  0.82             0.22                 0.84   
max                  1.66            14.09                 1.77   

       Years_in_Business  Prior_Claims_Count  Deductible_INR  
count            5025.00             5025.00         5025.00  
mean               -0.00                0.00           -0.00  
std                 1.00                1.00            1.00  
min                -1.73               -0.77           -0.72  
2

In [ ]:
with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("encoder.pkl saved")
print("scaler.pkl saved")
print()


encoder.pkl saved
scaler.pkl saved

These files are critical -- download them now from the left sidebar.
Without them, live submissions cannot be preprocessed the same way.


In [ ]:
df.to_csv("clean_submissions.csv", index=False)

print("clean_submissions.csv saved.")
print("Shape:", df.shape)
print()
print("Download this file too -- Student 2 trains their GLM models on it.")

clean_submissions.csv saved.
Shape: (5025, 59)

Download this file too -- Student 2 trains their GLM models on it.


In [ ]:
preprocessing_code = '''
import pandas as pd
import pickle

cat_cols = ["Occupancy", "Construction_Type", "Industry_Type",
            "Sprinkler_System", "Fire_Hydrant_Onsite", "Requested_Coverage"]

num_cols = ["Building_Age_Years", "Sum_Insured_INR", "Number_of_Employees",
            "Years_in_Business", "Prior_Claims_Count", "Deductible_INR"]

def preprocess_submission(df):
    """
    Call this on any new live submission from Student 4.
    Loads the saved encoder and scaler -- never re-fits them.
    Input:  raw dataframe with original column names
    Output: cleaned, encoded, scaled dataframe ready for models
    """
    df = df.copy()

    # Fill missing values
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")
    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # Load the saved encoder and scaler
    with open("encoder.pkl", "rb") as f:
        encoder = pickle.load(f)
    with open("scaler.pkl", "rb") as f:
        scaler = pickle.load(f)

    # Encode categorical columns
    cols_to_encode = [c for c in cat_cols if c in df.columns]
    encoded_array = encoder.transform(df[cols_to_encode])
    encoded_df = pd.DataFrame(
        encoded_array,
        columns=encoder.get_feature_names_out(cols_to_encode),
        index=df.index
    )
    df = df.drop(columns=cols_to_encode)
    df = pd.concat([df, encoded_df], axis=1)

    # Scale numeric columns
    cols_to_scale = [c for c in num_cols if c in df.columns]
    df[cols_to_scale] = scaler.transform(df[cols_to_scale])

    return df
'''

with open("preprocess_submission.py", "w") as f:
    f.write(preprocessing_code)

print("preprocess_submission.py saved.")
print("Download this file too -- Student 3 and Student 4 both import it.")

preprocess_submission.py saved.
Download this file too -- Student 3 and Student 4 both import it.
